# 04 · Reshape and transpose real images / Reshape y transposición de imágenes reales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669">PART III · EXERCISE · 15 MIN</span>

## Practise today / Practica hoy

Convert HWC to CHW and test that pixel values keep their meaning.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Convertir HWC a CHW y comprobar que los valores de píxeles conservan su significado.</div></div>

## Explore later / Explora después

Build image batches, compare NHWC with NCHW, and inspect memory layout.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Construir lotes de imágenes, comparar NHWC con NCHW e inspeccionar la disposición en memoria.</div></div>

Follow the core block immediately below. / Sigue el bloque esencial de abajo.

<!-- CORE-PATH -->
## Core path / Ruta esencial

Read and run this block from top to bottom: **recall → example → attempt → feedback → checkpoint**. Preparation and feedback definitions appear where needed. Try before opening a folded solution. Stop at **Core complete**; everything after it is **Explore later**.

🇪🇸 Lee y ejecuta este bloque de arriba abajo: **recuerda → ejemplo → intento → retroalimentación → comprobación**. La preparación y las funciones de comprobación aparecen donde se necesitan. Inténtalo antes de abrir una solución plegada. Detente en **Fin de la ruta esencial**; después empieza **Explora después**.

### Recall / Recuerda

Recall notebook 03: when a pixel column is standardized across images, what must stay paired with its pixel position?

🇪🇸 Recuerda el cuaderno 03: al estandarizar una columna de píxeles entre imágenes, ¿qué debe seguir asociado con su posición de píxel?

## Setup / Preparación

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Real images shipped with `scikit-image`: histology, microscopy, the astronaut,
and a cup of coffee.

No synthetic pixels anywhere in the exercises.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Imágenes reales incluidas en <code>scikit-image</code>: histología, microscopía, la astronauta y una taza de café. Ningún píxel sintético en los ejercicios.</div>

### Core prep 1/1 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from skimage import data

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Real images distributed with scikit-image.
photo = data.immunohistochemistry()   # (512, 512, 3), RGB histology
cells_img = data.cell()               # (660, 550), grayscale microscopy
astronaut = data.astronaut()          # (512, 512, 3), RGB photograph
coffee = data.coffee()                # (400, 600, 3), RGB photograph

def center_crop_rgb(img, size=256):
    # Deterministic centre crop so different real RGB images can be stacked.
    h, w, c = img.shape

    if c != 3 or h < size or w < size:
        raise ValueError(
            f"expected RGB image at least {size}x{size}, got {img.shape}"
        )

    r0 = (h - size) // 2
    c0 = (w - size) // 2

    return img[r0:r0 + size, c0:c0 + size]

rgb_sources = [photo, astronaut, coffee]
rgb_names = [
    "Histology / Histología",
    "Astronaut / Astronauta",
    "Coffee / Café",
]

print("Histology / Histología:", photo.shape)
print("Microscopy / Microscopía:", cells_img.shape)
print("Astronaut / Astronauta:", astronaut.shape)
print("Coffee / Café:", coffee.shape)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")


def as_channel_image(plane, channel):
    """One channel plane, rendered in its own colour rather than in grey.

    Not a colormap. `Reds` runs white -> red, so a bright pixel comes out dark
    and the picture reads inverted. Zeroing the other two channels is what the
    channel really looks like on its own, and it is one line of indexing.
    Toma un plano de un canal y lo devuelve en su propio color, poniendo a cero
    los otros dos.
    """
    out = np.zeros(plane.shape + (3,), dtype=plane.dtype)
    out[..., channel] = plane
    return out


## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Someone says the following. **Decide whether they are right before you
reveal anything** — commit to one answer, then open the check.

A tiny `(2, 2, 3)` image holding 0-11 becomes `(3, 2, 2)` two ways: `x.transpose(2, 0, 1)` and `x.reshape(3, 2, 2)`.

🇪🇸 Una imagen diminuta `(2, 2, 3)` con los valores 0-11 se convierte en `(3, 2, 2)` de dos formas: `x.transpose(2, 0, 1)` y `x.reshape(3, 2, 2)`.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,0.1);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div><div style="margin:.55em 0">&ldquo;Both results have the same shape, so <b>both are valid CHW images</b>.&rdquo;</div></div>

A prediction you have committed to is worth more than one you keep
adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Alguien afirma que, como ambos resultados tienen la <b>misma forma</b>, los dos son imágenes CHW válidas. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>

In [ ]:
#@title 🤔 Predict: does a matching shape prove it worked? / Predice: ¿basta con que la forma coincida? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
import numpy as np

pred_hwc = np.arange(12).reshape(2, 2, 3)
pred_right = pred_hwc.transpose(2, 0, 1)
pred_wrong = pred_hwc.reshape(3, 2, 2)

assert pred_right.shape == pred_wrong.shape
assert pred_right[1, 0, 0] == pred_hwc[0, 0, 1] == 1
assert pred_wrong[1, 0, 0] == 4
assert np.array_equal(pred_right.transpose(1, 2, 0), pred_hwc)
# --- end counterexample / fin del contraejemplo ---

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown. Four bilingual answers squeezed into
# one 640px line were hard to read, and a dropdown hides three of them until
# you open it -- the wrong shape for a question whose whole point is weighing
# the options against each other. One per line, with room around them.
# Botones de opción en vez de un desplegable: una respuesta por línea.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#059669"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence.

    A reading wants monospace and tight rows so the numbers line up under one
    another; a sentence wants prose type and room. The two used to share one
    13px monospace column, which is most of why the reveal read as a wall.
    """
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed.

    Exactly the same words: `check_prediction` still prints, and this catches
    what it printed and gives it typography. EN and ES stay written out as
    tags rather than becoming a colour, because a reader who cannot see the
    colour still has to be able to tell the two apart.
    """
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))

pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Both are valid — the shapes match / Ambas válidas — las formas coinciden", "both"),
        ("Neither is valid / Ninguna es válida", "neither"),
        ("Only transpose is valid / Solo la transposición es válida", "transpose_only"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)

def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba\n      cuando quieras.".replace("\n      ", " "))
        return

    print("Shapes / Formas:", pred_right.shape, "==", pred_wrong.shape)
    print()
    print("Channel 1, pixel (0,0) should be / El canal 1, píxel (0,0) debe ser:",
          pred_hwc[0, 0, 1])
    print("  transpose gives / la transposición da:", pred_right[1, 0, 0])
    print("  reshape gives / el reshape da:       ", pred_wrong[1, 0, 0])
    print()
    if choice == "transpose_only":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: the shape check passes for both; the channel-value check fails for reshape. Use transpose to reorder axes.")
    print("ES: la comprobación de forma pasa en ambos; la de valor del canal falla con reshape. Usa transpose para reordenar ejes.")

# The one <style> block in these notebooks, and the markdown rule does not
# cover it. ipywidgets gives no way to set the space between radio options
# from Python, and this is *widget output*, not a markdown cell: Colab strips
# <style> from markdown -- which is why every box in these notebooks is
# inline-styled -- but renders it in an output, the same path pandas' own
# Styler uses. Scoped to one added class so it can reach nothing else, and if
# it is ever dropped the options still work, just closer together.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

## Four axis letters / Cuatro letras de ejes

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

One tiny image, two layouts. The four letters below name every axis in the layouts you will meet in this notebook.

🇪🇸 Una imagen diminuta, dos disposiciones. Las cuatro letras de abajo nombran cada eje de las disposiciones de este cuaderno.

| Letter / Letra | English | Español |
|---|---|---|
| `N` | number of examples / batch | número de ejemplos / lote |
| `H` | height | alto |
| `W` | width | ancho |
| `C` | colour channels | canales de color |

Read every convention as a sentence.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE FOUR CONVENTIONS · LAS CUATRO CONVENCIONES</div><div style="margin:.55em 0"><code>HWC</code> — height × width × colour</div><div style="margin:.55em 0"><code>CHW</code> — colour × height × width</div><div style="margin:.55em 0"><code>NHWC</code> — examples × height × width × colour</div><div style="margin:.55em 0"><code>NCHW</code> — examples × colour × height × width</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0"><code>HWC</code> — alto × ancho × color</li><li style="margin:.35em 0"><code>CHW</code> — color × alto × ancho</li><li style="margin:.35em 0"><code>NHWC</code> — ejemplos × alto × ancho × color</li><li style="margin:.35em 0"><code>NCHW</code> — ejemplos × color × alto × ancho</li></ul></div>

## NHWC or NCHW / NHWC o NCHW

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

One framework expects `NHWC`. Another expects `NCHW`. A tensor can hold every
correct number and still be read wrongly, because the axes sit in the wrong
places.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">THE DANGEROUS PART · LO PELIGROSO</div>There is <b>no error message</b>. The code runs. The meaning does not.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un framework espera <code>NHWC</code> y otro <code>NCHW</code>. Un tensor puede contener todos los números correctos y aun así interpretarse mal.<br><br>Lo peligroso: <b>no hay mensaje de error</b>. El código corre; el significado no.</div>

### Example / Ejemplo

A `(2, 3)` table with row and column axes becomes `(3, 2)` with `M.transpose(1, 0)`: new axis 0 is old axis 1. The value `M[1, 2]` moves to `[2, 1]`. For an image, list the old axes in the new order the same way.

🇪🇸 Una tabla `(2, 3)` con ejes de filas y columnas pasa a `(3, 2)` con `M.transpose(1, 0)`: el nuevo eje 0 es el antiguo eje 1. El valor `M[1, 2]` pasa a `[2, 1]`. Para una imagen, enumera los ejes antiguos en el nuevo orden de la misma manera.

### One image, three planes of numbers / Una imagen, tres planos de números

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Twelve pixels, small enough to check by hand. Every channel value in them is 0, 128 or 255, so the second frame is verifiable rather than merely plausible: the red plane says 255 wherever the pixel looks red. The third frame is `HWC → CHW`, and axis 0 really does become colour — you can see the three planes are red, green and blue. The fourth is the mistake this notebook exists for.

$$
\bigl(A^{(2,0,1)}\bigr)_{c,h,w} = A_{h,w,c}
\qquad\qquad
\mathrm{flat}\bigl[\,h \cdot WC + w \cdot C + c\,\bigr] = A_{h,w,c}
$$

Read it as: a transpose is a rule about which index is read first, so every
number moves to wherever the new index order puts it. The flat offset on the
right is what `reshape` keeps untouched — it re-brackets that one buffer and
does nothing else. So the colour channels stay interleaved every third number,
and that interleaving is the red-green-blue stripe in the last frame.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-04-rgb.gif" alt="An animation of a three by four colour image of twelve flat saturated pixels. It is split into its red, green and blue planes, each drawn in its own colour with the channel value written in every cell. The image is then transposed to colour by height by width, drawn as a pile of one red, one green and one blue plane. Finally the transposed red plane, entirely red, is shown beside the reshaped plane of the same shape, which is striped red, green and blue because it took every third number from the buffer." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:11px">🇪🇸 ESPAÑOL</div>Doce píxeles, pocos como para comprobarlos a mano. Todos los valores de canal son 0, 128 o 255, así que el segundo fotograma se puede verificar en vez de solo creer: el plano rojo dice 255 dondequiera que el píxel se vea rojo. El tercero es <code>HWC → CHW</code>, y el eje 0 pasa realmente a ser el color: se ven los tres planos rojo, verde y azul. El cuarto es el error para el que existe este cuaderno.</div>

## Exercise 1 — one real image, two orderings / Ejercicio 1 — una imagen real, dos órdenes

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

`photo` is a real RGB histology image, <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(H, W, C) = (512, 512, 3)</span>.

`cells_img` is a real grayscale microscopy image, <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(H, W) = (660, 550)</span>.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Which image has a colour axis?</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>What should <code>photo.shape</code> become after <code>HWC → CHW</code>?</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>Which old axis becomes the new axis 0?</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> ¿cuál imagen tiene eje de color? <b>2 ·</b> ¿qué forma tendrá <code>photo</code> tras <code>HWC → CHW</code>? <b>3 ·</b> ¿qué eje antiguo se convierte en el nuevo eje 0?</div>

### Optional hints / Pistas opcionales

Try first; open one hint at a time. / Inténtalo primero; abre una pista a la vez.

<details>
<summary>Hint 1 / Pista 1</summary>

Write the old axis numbers under H, W, C. Read those numbers in the requested output order.

🇪🇸 Escribe los números de ejes originales bajo H, W, C. Lee esos números en el orden de salida pedido.

</details>

<details>
<summary>Hint 2 / Pista 2</summary>

The tuple passed to np.transpose lists old axes in their new order. Trace one pixel-channel value and then compare the full round trip.

🇪🇸 La tupla de np.transpose enumera los ejes antiguos en su nuevo orden. Sigue un valor de píxel y canal y después compara toda la ida y vuelta.

</details>



In [ ]:
# Feedback helper / Función de comprobación — run before your attempt / ejecuta antes del intento
import numpy as np

def check_core_answer(photo, chw):
    """Check shape and every pixel, including square images / Comprueba forma y píxeles."""
    photo, chw = map(np.asarray, (photo, chw))
    assert photo.ndim == 3, "Input must be HWC / La entrada debe ser HWC."
    assert chw.shape == (photo.shape[2], photo.shape[0], photo.shape[1]), "Expected CHW shape / Se espera forma CHW."
    assert np.isfinite(chw).all(), "Pixels must remain finite / Los píxeles deben seguir siendo finitos."
    assert np.array_equal(np.moveaxis(chw, 0, 2), photo), "Shape alone is insufficient: trace a pixel / La forma no basta: sigue un píxel."
    return "Checks passed; name the old axes in new order / Comprobaciones superadas; nombra los ejes antiguos en el nuevo orden."


### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** Where will photo[10, 20, 1] move in CHW?
2. **Run.** Complete Exercise 1.
3. **Explain.** Explain why transpose reorders axes.
4. **Check.** Run `check_core_answer(photo, chw)` on your own results before opening the solution. Check chw[1, 10, 20] == photo[10, 20, 1], then transpose back and compare every value.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** ¿Dónde quedará photo[10, 20, 1] en CHW?
2. **Ejecuta.** Completa el Ejercicio 1.
3. **Explica.** Explica por qué transpose reordena ejes.
4. **Comprueba.** Ejecuta `check_core_answer(photo, chw)` con tus resultados antes de abrir la solución. Comprueba chw[1, 10, 20] == photo[10, 20, 1]; luego invierte la transposición y compara todos los valores.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Print photo.shape and cells_img.shape.
# 2. Explain why cells_img has no colour axis.
#
# ES:
# 1. Imprime photo.shape y cells_img.shape.
# 2. Explica por qué cells_img no tiene eje de color.
#
# TODO 2 / TAREA 2
#
# EN:
# Convert photo from (H, W, C) to (C, H, W) with np.transpose.
# Name every output axis.
#
# ES:
# Convierte photo de (H, W, C) a (C, H, W) con np.transpose.
# Nombra cada eje de salida.
# Check your own results before opening the solution / Comprueba tus resultados antes de abrir la solución:
# check_core_answer(photo, chw)


In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

print("photo / foto:", photo.shape)
print("cells_img / microscopía:", cells_img.shape)
print()

print("EN: photo has three axes: height, width, colour.")
print("ES: photo tiene tres ejes: alto, ancho y color.")
print("EN: cells_img is grayscale, so it stores only height and width.")
print("ES: cells_img es en escala de grises, por eso solo almacena alto y ancho.")
print()

chw = np.transpose(photo, (2, 0, 1))

print("HWC:", photo.shape)
print("CHW:", chw.shape)
print()
print("EN: new axis 0 <- old axis 2 (colour)")
print("ES: nuevo eje 0 <- eje antiguo 2 (color)")
print("EN: new axis 1 <- old axis 0 (height)")
print("ES: nuevo eje 1 <- eje antiguo 0 (alto)")
print("EN: new axis 2 <- old axis 1 (width)")
print("ES: nuevo eje 2 <- eje antiguo 1 (ancho)")

assert np.array_equal(chw, np.moveaxis(photo, 2, 0))
print(check_core_answer(photo, chw))


<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

`np.transpose(photo, (2, 0, 1))` does not mean “make a shape `(3, 512, 512)`”.

It means: **put old axis 2 first, then old axis 0, then old axis 1**.

The original order was `(H, W, C)`, so the new one is `(C, H, W)`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>np.transpose(photo, (2, 0, 1))</code> no significa «crea una forma <code>(3, 512, 512)</code>». Significa: <b>coloca primero el eje antiguo 2, después el 0 y por último el 1</b>. El orden original era <code>(H, W, C)</code>, así que el nuevo es <code>(C, H, W)</code>.</div>

</details>

### Group discussion / Discusión en grupo

**Time:** 6 minutes.

A teammate converts NHWC images to NCHW using `reshape`. The output has the
requested shape. Design a review that can tell whether pixel meaning survived.

- What could a shape check miss?
- Which pixel or channel would you track through the conversion?
- What test would still work when two axes have the same size?

**Share:** One test in code or pseudocode and one visual check. Explain what
each catches. Try them on the notebook's correct and incorrect conversions.

<details>
<summary>Español</summary>

**Tiempo:** 6 minutos.

Alguien convierte imágenes NHWC a NCHW con `reshape`. La salida tiene la forma
pedida. Diseñen una revisión que compruebe si se conservó el significado de los
píxeles.

- ¿Qué error podría escapar a una comprobación de forma?
- ¿Qué píxel o canal seguirían durante la conversión?
- ¿Qué prueba funcionaría aunque dos ejes tuvieran el mismo tamaño?

**Compartan:** Una prueba en código o pseudocódigo y una comprobación visual.
Expliquen qué detecta cada una. Pruébenlas con ambas conversiones del cuaderno.

</details>

### Checkpoint / Comprobación

Record the group’s pixel-value test for HWC to CHW. Would checking only shape catch a mistaken reshape?

🇪🇸 Anota la prueba grupal de valores de píxeles para HWC a CHW. ¿Comprobar solo la forma detectaría un reshape incorrecto?

Answer / Respuesta: ___

<details>
<summary>Check after attempting / Comprueba después de intentarlo</summary>

Check `chw[c, h, w] == photo[h, w, c]` at distinctive pixels, or compare all entries with a transpose reference. A matching shape alone cannot establish this.

🇪🇸 Comprueba `chw[c, h, w] == photo[h, w, c]` en píxeles distintivos o compara todas las entradas con una transposición de referencia. Una forma coincidente no basta.

</details>

## Core complete / Fin de la ruta esencial

Keep your prediction, evidence, and explanation. Follow the facilitator’s quiz and break schedule before continuing.

🇪🇸 Guarda tu predicción, evidencia y explicación. Sigue las pausas y quizzes del facilitador antes de continuar.

The voice that turned to noise at the start of the session was this same bug: a reshape where a transpose was needed. See it on the [audio tensor stage](https://project-delphi.github.io/tensors-workshop/interactive/voice-stage.html?lang=en#scramble).

🇪🇸 La voz que se volvió ruido al comienzo de la sesión era este mismo error: un reshape donde hacía falta una transposición. Míralo en el [escenario del tensor de audio](https://project-delphi.github.io/tensors-workshop/interactive/voice-stage.html?lang=es#scramble).

[Kahoot 1](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-1) · [Next: Notebook 05 / Siguiente: cuaderno 05](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb).

## Explore later / Explora después

Optional reference, exercises, and explorers. These are outside this section’s live core. Continue in order when studying them; some reuse earlier setup.

🇪🇸 Material de consulta, ejercicios y exploradores opcionales. Quedan fuera de la ruta esencial en vivo. Continúa en orden al estudiarlos; algunos reutilizan la preparación anterior.

### The three images the batch is made of / Las tres imágenes del lote

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

`(3, 256, 256, 3)` is four numbers, and the first of them is these three photographs. They are deliberately unalike — a stained histology slide, a portrait, a cup of coffee — because a batch axis counts things that need not resemble each other at all.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>(3, 256, 256, 3)</code> son cuatro números, y el primero de ellos es estas tres fotografías. Son deliberadamente distintas — una muestra de histología teñida, un retrato, una taza de café — porque un eje de lote cuenta cosas que no tienen por qué parecerse en nada.</div>

In [ ]:
# The batch, recomputed here rather than taken from the folded solution, so
# this runs whether or not you opened it.
preview = np.stack([center_crop_rgb(img) for img in rgb_sources])

fig, axes = plt.subplots(1, len(preview), figsize=(10, 3.4))
for ax, image, title in zip(axes, preview, rgb_names):
    ax.imshow(image)
    ax.set_title(f"{title}\n{image.shape}")
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Batch / Lote:", preview.shape)
print("EN: N=3 images, H=256, W=256, C=3 — the batch axis counts photographs.")
print("ES: N=3 imágenes, H=256, W=256, C=3 — el eje de lote cuenta fotografías.")


### Convention translator / Traductor de convenciones

Pick a convention. Read what every position means.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige una convención y observa qué significa cada posición.</div>

In [ ]:
#@title 🔤 Convention translator / Traductor de convenciones — run me / ejecútame { display-mode: 'form' }

convention = widgets.ToggleButtons(
    options=["HWC", "CHW", "NHWC", "NCHW"],
    value="HWC",
    description="Convention / Convención:",
    style={"description_width": "145px"},
)

def explain_convention(value):
    meanings = {
        "HWC": (
            "(H, W, C)",
            "height × width × colour",
            "alto × ancho × color",
        ),
        "CHW": (
            "(C, H, W)",
            "colour × height × width",
            "color × alto × ancho",
        ),
        "NHWC": (
            "(N, H, W, C)",
            "examples × height × width × colour",
            "ejemplos × alto × ancho × color",
        ),
        "NCHW": (
            "(N, C, H, W)",
            "examples × colour × height × width",
            "ejemplos × color × alto × ancho",
        ),
    }

    symbols, en, es = meanings[value]

    print("Symbols / Símbolos:", symbols)
    print("EN:", en)
    print("ES:", es)

convention_output = widgets.interactive_output(
    explain_convention,
    {"value": convention},
)

display(widgets.VBox([convention, convention_output]))

### HWC ↔ CHW explorer / Explorador HWC ↔ CHW

Pick a colour channel. The left panel reads it from the original `HWC`. The
right panel reads **the same measured channel** from the transposed `CHW`.

If the transpose was right, the two are identical.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un canal: el panel izquierdo lo lee desde <code>HWC</code>, el derecho lee <b>el mismo canal medido</b> desde <code>CHW</code>. Si la transposición es correcta, ambos coinciden exactamente.</div>

In [ ]:
#@title 🖼️ HWC ↔ CHW explorer / Explorador HWC ↔ CHW — run me / ejecútame { display-mode: 'form' }

# The HWC->CHW transpose (Exercise 1, TODO 2), recomputed here so this
# explorer runs whether or not the folded solution was executed.
chw = np.transpose(photo, (2, 0, 1))

channel_selector = widgets.ToggleButtons(
    options=[
        ("R · Red / Rojo", 0),
        ("G · Green / Verde", 1),
        ("B · Blue / Azul", 2),
    ],
    value=0,
    description="Channel / Canal:",
    style={"description_width": "120px"},
)

def compare_hwc_chw(channel):
    channel_names = {
        0: "Red / Rojo",
        1: "Green / Verde",
        2: "Blue / Azul",
    }

    from_hwc = photo[:, :, channel]
    from_chw = chw[channel]

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))

    axes[0].imshow(as_channel_image(from_hwc, channel))
    axes[0].set_title(
        f"HWC → photo[:, :, {channel}]\n{channel_names[channel]}"
    )

    axes[1].imshow(as_channel_image(from_chw, channel))
    axes[1].set_title(
        f"CHW → chw[{channel}]\n{channel_names[channel]}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        "Same measured values / Mismos valores medidos:",
        np.array_equal(from_hwc, from_chw),
    )
    print("EN: transpose moved the colour axis; it did not change the pixel values.")
    print("ES: transpose movió el eje de color; no cambió los valores de los píxeles.")

channel_output = widgets.interactive_output(
    compare_hwc_chw,
    {"channel": channel_selector},
)

display(widgets.VBox([channel_selector, channel_output]))

## 4.2 Add a batch axis / Agrega un eje de lote

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

One RGB image is `(H, W, C)`. A batch of them adds an axis: `(N, H, W, C)`,
where `N` says **which image**.

We build a real batch from histology, the astronaut and the coffee. Their
original sizes differ, so each contributes a deterministic `256 × 256` centre
crop.

No pixel value is invented.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una imagen RGB es <code>(H, W, C)</code>; un lote añade un eje, <code>(N, H, W, C)</code>, donde <code>N</code> dice <b>qué imagen</b>. Como los tamaños originales difieren, tomamos un recorte central real de <code>256 × 256</code> de cada una. No se inventa ningún píxel.</div>

## Exercise 2 — a real batch, two axes of size 3 / Ejercicio 2 — un lote real, dos ejes de tamaño 3

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Stacked: <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">NHWC = (3, 256, 256, 3)</span>. Transposed:
<span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">NCHW = (3, 3, 256, 256)</span>.

Two axes now have size 3.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">SAME NUMBER, DIFFERENT QUESTION · MISMO NÚMERO, OTRA PREGUNTA</div><div style="margin:.55em 0"><b>axis 0</b> — which of the three <b>images</b>?</div><div style="margin:.55em 0"><b>axis 1</b> — which of the three <b>colour channels</b>?</div></div>

The number `3` carries none of that.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Dos ejes tienen tamaño <code>3</code>: uno responde «¿cuál de las tres imágenes?» y el otro «¿cuál de los tres canales?». El número por sí solo no lo dice.</div>

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Centre-crop every image in rgb_sources to 256×256.
# 2. Stack them into one batch.
# 3. Verify shape (3, 256, 256, 3).
# 4. Explain the meaning of N, H, W, and C.
#
# ES:
# 1. Recorta al centro cada imagen de rgb_sources a 256×256.
# 2. Apílalas en un solo lote.
# 3. Verifica la forma (3, 256, 256, 3).
# 4. Explica el significado de N, H, W y C.
#
# TODO 4 / TAREA 4
#
# EN:
# Convert NHWC to NCHW.
# Explain why axis 0 and axis 1 both have size 3 but different meanings.
#
# ES:
# Convierte NHWC a NCHW.
# Explica por qué los ejes 0 y 1 tienen tamaño 3 pero significados diferentes.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

batch = np.stack([center_crop_rgb(img) for img in rgb_sources])

print("NHWC:", batch.shape)
print("EN: N=3 images, H=256, W=256, C=3 colour channels.")
print("ES: N=3 imágenes, H=256, W=256, C=3 canales de color.")
print()

nchw = np.transpose(batch, (0, 3, 1, 2))

print("NCHW:", nchw.shape)
print("EN: axis 0 = image; axis 1 = colour.")
print("ES: eje 0 = imagen; eje 1 = color.")
print()

one_image = nchw[0]
red_all_images = nchw[:, 0]

print("nchw[0].shape:", one_image.shape)
print("EN: one image, all three channels.")
print("ES: una imagen, sus tres canales.")
print()

print("nchw[:, 0].shape:", red_all_images.shape)
print("EN: red channel from all three images.")
print("ES: canal rojo de las tres imágenes.")

assert batch.shape == (3, 256, 256, 3)
assert nchw.shape == (3, 3, 256, 256)
assert np.array_equal(
    one_image,
    np.transpose(batch[0], (2, 0, 1)),
)
assert np.array_equal(
    red_all_images,
    batch[:, :, :, 0],
)

### Batch-axis explorer / Explorador de ejes del lote

Pick an image `N` and a channel `C`. The notebook reads the same data from
`NHWC` and from `NCHW`.

Both axes have size `3`. The controls keep their roles apart.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige una imagen <code>N</code> y un canal <code>C</code>: el cuaderno lee los mismos datos desde <code>NHWC</code> y desde <code>NCHW</code>. Ambos ejes miden <code>3</code>, y los controles mantienen separados sus papeles.</div>

In [ ]:
#@title 🖼️ Batch-axis explorer / Explorador de ejes del lote — run me / ejecútame { display-mode: 'form' }

# The real batch and its NHWC->NCHW transpose (Exercise 2), recomputed here
# so this explorer runs whether or not the folded solution was executed.
batch = np.stack([center_crop_rgb(img) for img in rgb_sources])
nchw = np.transpose(batch, (0, 3, 1, 2))

image_selector = widgets.ToggleButtons(
    options=[
        ("N=0 · Histology / Histología", 0),
        ("N=1 · Astronaut / Astronauta", 1),
        ("N=2 · Coffee / Café", 2),
    ],
    value=0,
    description="Image N / Imagen N:",
    style={"description_width": "130px"},
)

batch_channel_selector = widgets.ToggleButtons(
    options=[
        ("C=0 · Red / Rojo", 0),
        ("C=1 · Green / Verde", 1),
        ("C=2 · Blue / Azul", 2),
    ],
    value=0,
    description="Channel C / Canal C:",
    style={"description_width": "130px"},
)

def explore_batch_axes(image_idx, channel_idx):
    from_nhwc = batch[image_idx, :, :, channel_idx]
    from_nchw = nchw[image_idx, channel_idx, :, :]

    plt.close("all")
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))

    axes[0].imshow(batch[image_idx])
    axes[0].set_title(
        f"N={image_idx}\n{rgb_names[image_idx]}"
    )

    axes[1].imshow(from_nhwc, cmap="gray")
    axes[1].set_title(
        f"NHWC[{image_idx}, :, :, {channel_idx}]"
    )

    axes[2].imshow(from_nchw, cmap="gray")
    axes[2].set_title(
        f"NCHW[{image_idx}, {channel_idx}, :, :]"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        "Same measured plane / Mismo plano medido:",
        np.array_equal(from_nhwc, from_nchw),
    )
    print(f"N={image_idx} -> EN: which image | ES: qué imagen")
    print(f"C={channel_idx} -> EN: which colour | ES: qué color")

batch_output = widgets.interactive_output(
    explore_batch_axes,
    {
        "image_idx": image_selector,
        "channel_idx": batch_channel_selector,
    },
)

display(
    widgets.VBox([
        image_selector,
        batch_channel_selector,
        batch_output,
    ])
)

<details>
<summary><strong>Why two size-3 axes are different / Por qué dos ejes de tamaño 3 son diferentes</strong></summary>

In `(N, C, H, W) = (3, 3, 256, 256)` the first `3` answers *which of the three
images*, and the second answers *which of the three RGB channels*.

A shape records sizes. It does not record labels.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>En <code>(N, C, H, W) = (3, 3, 256, 256)</code>, el primer <code>3</code> responde «¿cuál de las tres imágenes?» y el segundo «¿cuál de los tres canales RGB?». La forma guarda tamaños, no etiquetas.</div>

</details>

## 4.3 Reshape vs. transpose / Reshape vs. transpose

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

The most important part of this notebook.

From `photo.shape = (512, 512, 3)`, both of these produce `(3, 512, 512)`:

- `np.transpose(photo, (2, 0, 1))` — correct;
- `photo.reshape(3, 512, 512)` — same shape, scrambled image.

Read the elements out in order and the difference is exact.

$$
\operatorname{reshape}: \quad A_{ij} \mapsto \mathrm{flat}\bigl[i \cdot n + j\bigr]
\qquad\qquad
\operatorname{transpose}: \quad \bigl(A^{\mathsf T}\bigr)_{ji} = A_{ij}
$$

Read it as: `reshape` keeps the flat order and only changes where the brackets
go, so `a.reshape(...).ravel()` is always `a.ravel()`.
`transpose` changes which index is read first, so its `ravel` is a permutation
of the same numbers — same values, different order.


<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE WHOLE DISTINCTION · TODA LA DIFERENCIA</div><div style="margin:.55em 0"><b>reshape</b> changes the shape and nothing else. The flat sequence is untouched: <code>a.reshape(...).ravel()</code> always equals <code>a.ravel()</code>. It moves the brackets.</div><div style="margin:.55em 0"><b>transpose</b> changes the shape <i>and</i> reorders that flat sequence. <code>a.transpose(...).ravel()</code> is a permutation of <code>a.ravel()</code> — same values, different reading order.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0"><b>reshape</b> cambia la forma y nada más: <code>a.reshape(...).ravel()</code> siempre es igual a <code>a.ravel()</code>.</li><li style="margin:.35em 0"><b>transpose</b> cambia la forma <i>y</i> reordena esa secuencia plana: <code>a.transpose(...).ravel()</code> es una permutación de <code>a.ravel()</code>.</li></ul></div>

### Transpose against reshape / Transpose frente a reshape

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Both produce shape <code>(5, 3, 4)</code>, and they are not the same tensor. The last frame puts them side by side and marks one position: <code>[1, 0, 0]</code> holds 1 after the transpose and 12 after the reshape. A shape check passes for both.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-04-transpose-vs-reshape.gif" alt="An animation comparing transpose and reshape on the same tensor. Both give shape 5 by 3 by 4. The final frame shows the two results side by side with the cell at position 1, 0, 0 highlighted in each: it holds 1 after the transpose and 12 after the reshape." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ambas producen la forma <code>(5, 3, 4)</code>, y no son el mismo tensor. El último fotograma las pone lado a lado y marca una posición: <code>[1, 0, 0]</code> contiene 1 tras la transposición y 12 tras el reshape. Una comprobación de forma pasa en ambos casos.</div>

### The order the buffer is read in / El orden en que se lee el búfer

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Six numbers, because the claim is about order and order is only checkable if you can hold every number at once. `reshape` reads along the rows and hands them back unchanged; `transpose` permutes the strides, so its `ravel` comes out reordered.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-04-ravel.gif" alt="An animation showing a 2 by 3 array of 0 to 5, its ravel as 0 1 2 3 4 5, its transpose as a 3 by 2 array, and that transpose's ravel as 0 3 1 4 2 5." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Seis números, porque la afirmación trata del orden y el orden solo se puede comprobar si puedes retener todos los números a la vez. <code>reshape</code> lee a lo largo de las filas y los devuelve sin cambios; <code>transpose</code> permuta los strides, así que su <code>ravel</code> sale reordenado.</div>

The three photographs of this notebook are cubes, one per byte, on the [image tensor visualizer](https://project-delphi.github.io/tensors-workshop/interactive/image-tensor.html?lang=en). Transpose NHWC to NCHW and watch the shape and the strides permute while the picture stays put; then compare with reshape and watch it break.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Las tres fotografías de este cuaderno son cubos, uno por byte, en el <a href="https://project-delphi.github.io/tensors-workshop/interactive/image-tensor.html?lang=es">visualizador del tensor de imagen</a>. Transpón NHWC a NCHW y observa cómo la forma y los strides se permutan mientras la imagen no se mueve; después compáralo con reshape y mira cómo se rompe.</div>

A reshape has a second trap once axes are given a meaning, like the heads in a multi-head attention layer: reshaping (4, 8) into (2, 4, 4) is a valid reshape whichever axis order you pick, but only one of them keeps a head's rows the tokens they started as. The attention stage's [heads picture](https://project-delphi.github.io/tensors-workshop/interactive/attention-stage.html?lang=en#heads) is the honest reshape-then-transpose beside the direct one that mixes tokens instead.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un reshape tiene una segunda trampa en cuanto los ejes tienen un significado, como las cabezas de una capa de atención multi-cabeza: convertir (4, 8) en (2, 4, 4) es un reshape válido con cualquier orden de ejes que elijas, pero solo uno conserva en las filas de una cabeza los tokens de los que partieron. La <a href="https://project-delphi.github.io/tensors-workshop/interactive/attention-stage.html?lang=es#heads">imagen de cabezas</a> del escenario de atención es el reshape honesto seguido de transpose junto al directo que mezcla tokens.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The animations above loop forever and a GIF cannot
# be paused — so this fetches the same frames and hands them over one at a
# time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-04-transpose-vs-reshape.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-04-ravel.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-04-rgb.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))


### One implementation note / Una nota de implementación

NumPy does not move memory to transpose. It hands back a view with permuted
**strides**. The buffer is untouched, and the reordering is paid for later — by
the first operation that forces a copy: `ravel`, `copy`, `ascontiguousarray`,
or a library demanding contiguous input.

Free to write. Not always free to run.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE LIBRARY · LA BIBLIOTECA</div>Books arranged as <b>shelf × position × category</b>. <code>transpose</code> moves the labelled dimensions. <code>reshape</code> takes the books in their current reading order and fills a differently shaped unit, labels be damned.</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>NumPy no mueve memoria al transponer: devuelve una vista con <b>strides</b> permutados. El reordenamiento se paga después, en la primera operación que obliga a copiar — <code>ravel</code>, <code>copy</code>, <code>ascontiguousarray</code>, o una librería que exija datos contiguos.<br><br>Libros ordenados por <b>estante × posición × categoría</b>: <code>transpose</code> mueve las dimensiones etiquetadas; <code>reshape</code> toma los libros en su orden actual y llena una estantería de otra forma, ignorando las etiquetas.</div>

In [ ]:
# The claim above, in six values. Deliberately synthetic and tiny: the point is
# the reading order, and 0..5 is the only data you can check by eye.
a = np.arange(6).reshape(2, 3)
print("a =\n", a, "\n")

print("a.ravel()               ", a.ravel())              # 0 1 2 3 4 5
print("a.reshape(3, 2).ravel() ", a.reshape(3, 2).ravel())  # unchanged
print("a.T.ravel()             ", a.T.ravel())            # reordered
print()
print("shapes  a.reshape(3, 2):", a.reshape(3, 2).shape, " a.T:", a.T.shape)
print("strides a:", a.strides, " a.T:", a.T.strides,
      "— transpose permuted the strides, not the buffer")

## Exercise 3 — code that runs and is still wrong / Ejercicio 3 — código que funciona y aun así está mal

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div>Two arrays share the shape <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(5,150,105,.14);border:1px solid rgba(5,150,105,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(3, 512, 512)</span>.<div style='margin-top:10px'>Does that guarantee <code>array_a[0]</code> and <code>array_b[0]</code> are the same colour channel?</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Dos arreglos tienen la misma forma <code>(3, 512, 512)</code>. ¿Eso garantiza que <code>array_a[0]</code> y <code>array_b[0]</code> sean el mismo canal de color?</div>

In [ ]:
# TODO 5 / TAREA 5
#
# EN:
# 1. Compute the correct CHW array with transpose.
# 2. Compute photo.reshape(3, 512, 512).
# 3. Verify that the shapes are equal.
# 4. Verify whether the arrays themselves are equal.
# 5. Display plane 0 from both arrays.
# 6. Explain why reshape cannot replace transpose here.
#
# ES:
# 1. Calcula el arreglo CHW correcto con transpose.
# 2. Calcula photo.reshape(3, 512, 512).
# 3. Verifica que las formas sean iguales.
# 4. Verifica si los arreglos completos son iguales.
# 5. Muestra el plano 0 de ambos.
# 6. Explica por qué reshape no puede sustituir transpose en este caso.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

correct_chw = np.transpose(photo, (2, 0, 1))
wrong_reshape = photo.reshape(3, 512, 512)

print("Correct CHW / CHW correcto:", correct_chw.shape)
print("Reshape output / Salida reshape:", wrong_reshape.shape)
print()

print(
    "Same shape / Misma forma:",
    correct_chw.shape == wrong_reshape.shape,
)
print(
    "Same data arrangement / Misma organización de datos:",
    np.array_equal(correct_chw, wrong_reshape),
)

difference_fraction = np.mean(correct_chw != wrong_reshape)

print(
    f"Different positions / Posiciones diferentes: "
    f"{difference_fraction:.1%}"
)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))

axes[0].imshow(photo)
axes[0].set_title("Original RGB / RGB original")

axes[1].imshow(as_channel_image(correct_chw[0], 0))
axes[1].set_title(
    "transpose\nreal red channel / canal rojo real"
)

axes[2].imshow(as_channel_image(wrong_reshape[0], 0))
axes[2].set_title(
    "reshape\nscrambled interpretation / interpretación alterada"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print()
print("EN: transpose preserved the meaning of the colour axis.")
print("ES: transpose conservó el significado del eje de color.")
print("EN: reshape reached the same shape but did not move the colour axis correctly.")
print("ES: reshape alcanzó la misma forma, pero no movió correctamente el eje de color.")

### Transpose-vs-reshape explorer / Explorador transpose contra reshape

Pick a plane — `0`, `1` or `2` — and the method that produced
`(3, 512, 512)`. Compare against the true RGB channel.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un plano — <code>0</code>, <code>1</code> o <code>2</code> — y el método que produjo <code>(3, 512, 512)</code>. Compara con el canal RGB verdadero.</div>

In [ ]:
#@title 🔬 Transpose vs reshape / Transpose contra reshape — run me / ejecútame { display-mode: 'form' }

# The correct transpose and the same-shape reshape (Exercise 3), recomputed
# here so this explorer runs whether or not the folded solution was executed.
correct_chw = np.transpose(photo, (2, 0, 1))
wrong_reshape = photo.reshape(3, 512, 512)

plane_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=2,
    step=1,
    description="Plane / Plano:",
    continuous_update=False,
    style={"description_width": "100px"},
)

method_toggle = widgets.ToggleButtons(
    options=[
        ("Transpose / Transposición", "transpose"),
        ("Reshape", "reshape"),
    ],
    value="transpose",
    description="Method / Método:",
    style={"description_width": "110px"},
)

def explore_method(plane, method):
    true_channel = photo[:, :, plane]

    if method == "transpose":
        candidate = correct_chw[plane]
        method_name = "transpose / transposición"
    else:
        candidate = wrong_reshape[plane]
        method_name = "reshape"

    same = np.array_equal(candidate, true_channel)
    mae = float(
        np.mean(
            np.abs(
                candidate.astype(np.float32)
                - true_channel.astype(np.float32)
            )
        )
    )

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.5))

    axes[0].imshow(as_channel_image(true_channel, plane))
    axes[0].set_title(
        f"True channel {plane} / Canal real {plane}"
    )

    axes[1].imshow(as_channel_image(candidate, plane))
    axes[1].set_title(
        f"{method_name}\nplane/plano {plane}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print("Exact match / Coincidencia exacta:", same)
    print(f"Mean absolute difference / Diferencia absoluta media: {mae:.3f}")

    if same:
        print("EN: this operation preserved the intended channel semantics.")
        print("ES: esta operación conservó el significado correcto del canal.")
    else:
        print("EN: the shape is plausible, but the channel semantics are wrong.")
        print("ES: la forma parece correcta, pero el significado del canal es incorrecto.")

method_output = widgets.interactive_output(
    explore_method,
    {
        "plane": plane_slider,
        "method": method_toggle,
    },
)

display(
    widgets.VBox([
        widgets.HBox([plane_slider, method_toggle]),
        method_output,
    ])
)

## Which one do I want? / ¿Cuál necesito?

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">THE RULE · LA REGLA</div><div style="margin:.55em 0"><b>Move axes → transpose.</b> <code>HWC → CHW</code>, <code>NHWC → NCHW</code>: the semantic positions change.</div><div style="margin:.55em 0"><b>Group dimensions → reshape.</b> <code>(H, W) → (H×W,)</code>: you regroup on purpose, and no axis pretends to have moved.</div></div>

And a test you can actually run: **`reshape` never changes `ravel()`;
`transpose` always does.**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>Mover ejes → transpose. Agrupar dimensiones → reshape.</b> O como comprobación ejecutable: <code>reshape</code> nunca cambia <code>ravel()</code>, <code>transpose</code> siempre lo hace.</div>

## Quick reasoning challenge / Reto rápido de razonamiento

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

**Transpose** or **reshape**? Decide before you run.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">FOUR TASKS · CUATRO TAREAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><code>(512,512,3) → (3,512,512)</code>, because a model wants colour first.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><code>(8,8) → (64,)</code>, because one digit should become one feature vector.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><code>(3,256,256,3) → (3,3,256,256)</code>, because the framework wants channel before height.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><code>(1797,8,8) → (1797,64)</code>, because each image should become one row.</div></div>

The question is never only “what shape do I want?” It is **“do I want to move
axes, or group dimensions?”**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La pregunta nunca es solo «¿qué forma quiero?», sino <b>«¿quiero mover ejes o agrupar dimensiones?»</b></div>

In [ ]:
#@title 🧠 Reasoning challenge / Reto de razonamiento — run me / ejecútame { display-mode: 'form' }

reasoning_task = widgets.Dropdown(
    options=[
        ("1 · HWC → CHW", 1),
        ("2 · 8×8 image → 64 values / imagen 8×8 → 64 valores", 2),
        ("3 · NHWC → NCHW", 3),
        ("4 · (1797,8,8) → (1797,64)", 4),
    ],
    value=1,
    description="Task / Tarea:",
    style={"description_width": "100px"},
)

def explain_choice(task):
    answers = {
        1: (
            "transpose",
            "colour must move from the last axis to the first",
            "el color debe moverse del último eje al primero",
        ),
        2: (
            "reshape",
            "height and width are intentionally grouped into one feature axis",
            "alto y ancho se agrupan intencionalmente en un eje de características",
        ),
        3: (
            "transpose",
            "the colour axis must move before the spatial axes",
            "el eje de color debe moverse antes de los ejes espaciales",
        ),
        4: (
            "reshape",
            "each 8×8 image is intentionally flattened into 64 values",
            "cada imagen 8×8 se aplana intencionalmente en 64 valores",
        ),
    }

    operation, en, es = answers[task]

    print("Operation / Operación:", operation)
    print("EN:", en)
    print("ES:", es)

reasoning_output = widgets.interactive_output(
    explain_choice,
    {"task": reasoning_task},
)

display(widgets.VBox([reasoning_task, reasoning_output]))

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

Every value you touched was a measured pixel.

<div style="border-left:5px solid #059669;background:rgba(5,150,105,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#059669;margin-bottom:11px">FIVE IDEAS · CINCO IDEAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><b>Shape is not semantics.</b> Two axes of size <code>3</code> can mean different things.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><b>HWC and CHW</b> hold the same image in a different axis order.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><b>NHWC and NCHW</b> hold the same batch in a different axis order.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><b>Transpose moves axes.</b></div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#059669;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">5</span><b>Reshape regroups stored values.</b> It can hit the shape you wanted and the meaning you did not.</div></div>

Move axes when axis meaning changes. Reshape when you regroup on purpose — and
never let a reshape pretend an axis moved.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> la forma no es la semántica; <b>2 ·</b> <code>HWC</code> y <code>CHW</code> son la misma imagen en otro orden; <b>3 ·</b> igual con <code>NHWC</code> y <code>NCHW</code>; <b>4 ·</b> transpose mueve ejes; <b>5 ·</b> reshape reagrupa valores y puede acertar la forma y errar el significado.</div>

## Where this goes next / Adónde sigue esto

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

[A tensor in pure Python](https://project-delphi.github.io/ml-blog/posts/numpy-like-tensor/) builds `reshape` and `transpose` from nothing but a
flat list and a shape. Its `reshape` leaves the stored values in order; its
`transpose` has to reorder them. That is the difference this notebook drew with
pixels, with no library in the way. The post is in English.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La entrada de arriba construye <code>reshape</code> y <code>transpose</code> solo con una lista plana y una forma. Su <code>reshape</code> deja los valores guardados en orden; su <code>transpose</code> tiene que reordenarlos. Es la diferencia que este cuaderno mostró con píxeles, sin ninguna biblioteca de por medio. La entrada está en inglés.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#059669,rgba(5,150,105,0))"></div>

## Time for Kahoot 🎯 / Hora de Kahoot 🎯

**Kahoot 1 — Tensor Vocabulary & Shapes / Vocabulario de tensores y formas**  
6 questions / 6 preguntas · about 5 minutes / unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Entra a <b>kahoot.it</b> con el PIN que aparece en la pantalla del facilitador.</div></div>

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-1)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_1_vocabulary_shapes.xlsx)

Next / Siguiente: **05 · Video pipeline design / Diseño de un pipeline de vídeo** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)